# Planning:


### Required Data:
Profiles can be nodes in a graph, each containing the necessary data. They will each have a user id that can be accessed through a dictionary.
Connections, blocked users, and skills can be stored under the profile node, in a set.
User id, name, job title, employer, industry, and location can simply be variables under each profile node.
A disconnected group, an isolated user, and at least one blocked pair can be created when generating the graph.

### Required Data Structures:
Dictionary will be used when storing / searching for user ids.
Sets will be used for connections, blocked users, and skills.
Adjacency List Graph will be used for the profiles and connections.
Queue can be used to implement breadth first search for if two users are connected.
Heap can be used to implement / store top connection recommendations
**Note, the required extension should use a data structure other than the above five in some justifiable way**




In [98]:
import random
import math

In [99]:
class Profile:

    def __init__(self, user_id, name, location, job_title = "Unemployed", employer = "None", industry = "None"):

        self.user_id = user_id
        self.name = name
        self.location = location
        self.job_title = job_title
        self.employer = employer
        self.industry = industry
        self.connections = set()
        self.blocked_users = set()
        self.skills = set()

    def __str__(self):

        return f"User #{self.user_id}: {self.name} lives in {self.location} and works as a {self.job_title} for {self.employer} in the {self.industry} industry."

    def connection_print(self):

        to_print = f"Name: {self.name}. Connections: | "

        for c in self.connections:
            to_print += f"{c.name} | "

        print(to_print)
        return None


In [100]:
#Generic Functions
def shortest_connection_chain(start_user, target_user, print_it = False):

    if start_user == target_user:
        if print_it == True:
            print(f"That's the same user.")
        return((0,[start_user.name]))

    queue = []
    #Uses tuple to store the user Profile and the names of connections along the way
    current_node_path = (start_user,[start_user.name])
    queue.append(current_node_path)

    visited = set()

    found = False

    #Breadth First Search Loop
    while len(queue) != 0:

        #splits tuple into component parts for readability
        current_node = current_node_path[0]
        current_path = current_node_path[1]

        for c in current_node.connections:

            #This connection is the one we're looking for
            if c == target_user:
                current_path.append(c.name)
                found = True
                break

            #This connection needs to be searched later
            if c not in visited:
                new_path = current_path.copy()
                new_path.append(c.name)
                queue.append((c,new_path))

        #Prints / Returns the depth and path if we found the correct connection
        if found == True:
            depth = len(current_path) - 1
            if print_it == True:
                print(f"The shortest path between {start_user.name} and {target_user.name} is {str(depth)} connections. The path is {str(current_path)}")
            return((depth,current_path))

        #Only activates if the correct connection was not found. Remove from queue, add to visited, and change current_node_path
        visited.add(current_node)
        queue.pop(0)
        if len(queue) != 0:
            current_node_path = queue[0] 

    #We only get here if length of queue is 0, meaning no connection could be found
    if print_it == True:
        print(f"{start_user.name} and {target_user.name} are not connected.")
    return((None,None))

def connect(user,friends_list):

    for f in friends_list:
        user.connections.add(f)
        f.connections.add(user)

    return None

def id_name_print(dictionary):
    to_print = ""
    for id,user in dictionary.items():
        to_print += f"User #{str(id)} is {user.name}\n"
    print(to_print)
    return None

In [101]:
#Functions for generating dataset
def generate_profiles(number,user_dict):

    #Surnames chosen from most common surnames in the united states according to census.gov
    surnames = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Rodriguez","Davis","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson"]
    #Given names not chosen that way
    given_names = ["Tom","Tommy","Thomas","Thomathan","Thomicheal","Thomater","Thomeven","Thomephen","Thouglas","Thombert","Orion"]
    #Cities / Locations chosen from the most populous cities in the united states according to census.gov, with the word "city" removed (who calls it 'Chicago city')
    locations = ["New York","Los Angeles","Chicago","Houston","Phoenix"]
    #industries are tuples with the name of the industry and some example jobs in that industry
    industries = [("Construction",["Construction Worker","Construction Manager"]),("Accounting",["Tax Accountant","Auditer","Private Accountant"]),("Tech",["Data Scientist","Software Developer","Obnoxious CEO"])]
    employers = ["Hegemonic Monopoly Inc.","Small Startup and Co."]
   

    for i in range(number):

        user_id = i
        name = f"{random.choice(given_names)} {random.choice(surnames)}"
        location = random.choice(locations)
        industry_list = random.choice(industries)
        industry = industry_list[0]
        job_title = random.choice(industry_list[1])
        employer = random.choice(employers)

        new_node = Profile(user_id,name,location,job_title,employer,industry)
        user_dict[i] = new_node

    return user_dict

def connect_the_dots(user_dict, minimum_connections = 35, isolated_users = 0, disconnected_triplets = 0):

    def isolate_them(ids):

            for i in range(isolated_users):
                isolated = random.choice(ids)
                ids.remove(isolated)
            return None

    def triplet_disconnection(ids):

        for k in range(disconnected_triplets):
        
                disconnected_ids = random.sample(ids,3)
        
                for y in disconnected_ids:
                    ids.remove(y)
                    friends = []
                    for x in disconnected_ids:
                         if x != y:
                              friends.append(user_dict[x])
                    connect(user_dict[y],friends)
                
        return None

    user_count = len(user_dict)
    minimum_users = isolated_users + (disconnected_triplets * 3)
    possible_connections = math.factorial(user_count - minimum_users) + (disconnected_triplets * 3)
    connection_count = 0

    #checking for errors
    if minimum_connections > possible_connections:
        raise Exception("Not possible to create the minimum number of connections")

    if user_count < minimum_users:
        raise Exception("Not enough users for that many isolated or disconnected groups!")

    user_ids = list(user_dict.keys())

    isolate_them(user_ids)

    triplet_disconnection(user_ids)

    connection_count += disconnected_triplets * 3

    #Stores the user_ids who are not isolated or in a triplet
    social_user_ids = user_ids.copy()

    #For each id, makes a list of who can't be connected, then randomly assigns some connections
    for id in user_ids:

        profile = user_dict[id]
        cant_friend = {profile}

        for c in profile.connections:

             cant_friend.add(c)

        for c in profile.blocked_users:
             cant_friend.add(c)

        potential_connections = user_count - minimum_users - len(cant_friend)
        #Note: Potential Friends divided by 4 to make less connections. If you desire more connections, decrease that number.
        number_of_friends = random.randint(0,potential_connections // 4)
        friends_list = set()

        while len(friends_list) < number_of_friends:
             potential_friend = user_dict[random.choice(user_ids)]
             if potential_friend not in cant_friend:
                  friends_list.add(potential_friend)
                  cant_friend.add(potential_friend)
                  connection_count += 1
                  
        connect(profile,friends_list)

    #randomly makes connections until minimum connections is satisfied
    while connection_count < minimum_connections:
         
         lonely_node = user_dict[random.choice(social_user_ids)]

         max_connections = len(social_user_ids) - 1 - len(lonely_node.blocked_users)

         if max_connections > len(lonely_node.connections):

            cant_friend = {lonely_node}
            potential_friend = lonely_node

            for friend in lonely_node.connections:
                cant_friend.add(friend)

            for blocked in lonely_node.blocked_users:
                 cant_friend.add(blocked)

            while potential_friend in cant_friend:
                 potential_friend = user_dict[random.choice(social_user_ids)]

            connect(lonely_node,[potential_friend])

    return None
            


In [102]:
user_dict = {}

generate_profiles(30,user_dict)
connect_the_dots(user_dict,35,3,1)

for user in user_dict.values():
    user.connection_print()

Name: Thombert Gonzalez. Connections: | 
Name: Thomas Garcia. Connections: | Thomephen Johnson | Thomeven Wilson | Tommy Brown | Thomicheal Wilson | Tommy Wilson | 
Name: Thomeven Davis. Connections: | Thomas Rodriguez | Thomeven Wilson | Thomas Miller | Thomephen Wilson | Thomas Hernandez | Thomephen Jones | Thombert Williams | Tommy Brown | Thomicheal Miller | 
Name: Tommy Wilson. Connections: | Thomas Rodriguez | Thomas Miller | Thomas Hernandez | Thomas Garcia | Tommy Johnson | Thomephen Smith | 
Name: Thombert Davis. Connections: | Thouglas Lopez | Thomas Gonzalez | 
Name: Thomas Hernandez. Connections: | Thomas Rodriguez | Thomas Miller | Thomephen Wilson | Thomicheal Martinez | Thomeven Davis | Thomeven Davis | Tommy Wilson | 
Name: Thomephen Jones. Connections: | Thomathan Lopez | Thomephen Johnson | Thomeven Davis | 
Name: Thomephen Smith. Connections: | Thomeven Wilson | Thomicheal Martinez | Tommy Wilson | 
Name: Thomicheal Anderson. Connections: | Thomathan Lopez | Thomeven